# 04 — Bengaluru / Karnataka STAC Search, Inference & Export

1. Searches Microsoft Planetary Computer STAC (`sentinel-1-rtc`) for the Bengaluru/Karnataka AOI (no hardcoded item IDs).
2. Signs assets via `planetary_computer.sign_inplace` and groups scenes by matching orbit direction & relative orbit.
3. Converts Sentinel-1 RTC linear power to dB (`10*log10(max(x, 1e-6))`) and compares its histogram against Sen1Floods11 dB.
4. Runs tiled U-Net inference, masks permanent water (`jrc-gsw`), cleans & polygonizes in `EPSG:32643` (UTM 43N), overlays `esa-worldcover` and OpenStreetMap assets.
5. Runs Sentinel-2 L2A (`sentinel-2-l2a`) before/after NDVI drop + BSI increase land-change detection.
6. Exports all real outputs (`sample: false`) into `backend/data/`.

In [ ]:
import json, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ml import config
from ml.src.stac_fetch import (
    search_sentinel1_scenes,
    select_before_after_pair,
    search_auxiliary_item,
)
from ml.src.preprocess import read_s1_rtc_db, reproject_asset_to_grid, read_s2_l2a_indices
from ml.src.infer import load_trained_unet, predict_flood_probability
from ml.src.postprocess import (
    clean_flood_mask,
    polygonize_mask,
    compute_impact_metrics,
    detect_land_change_polygons,
)
from ml.src.export_results import export_pipeline_outputs, save_sar_preview_png, save_rgb_preview_png
from ml.src.dataset import load_s1_chip, read_split_csv

In [ ]:
# 1. Search Planetary Computer STAC for flood & dry-reference S1 RTC scenes
flood_scenes = search_sentinel1_scenes(
    bbox=config.DEFAULT_AOI_BBOX,
    datetime_range=config.DEFAULT_DATE_RANGE,
    output_json_path=config.BACKEND_DATA_DIR / 'scenes.json',
)
dry_scenes = search_sentinel1_scenes(
    bbox=config.DEFAULT_AOI_BBOX,
    datetime_range=config.DRY_REFERENCE_RANGE,
)
if not flood_scenes:
    raise RuntimeError('No flood-window Sentinel-1 RTC scenes found. Adjust DEFAULT_AOI_BBOX or DEFAULT_DATE_RANGE in ml/config.py.')

before_scene, after_scene = select_before_after_pair(flood_scenes, dry_scenes)
print('Selected BEFORE scene:', before_scene['id'] if before_scene else 'None (using single post-event scene)')
print('Selected AFTER  scene:', after_scene['id'])

In [ ]:
# 2. Read VV & VH in linear power, convert to dB, and verify distribution vs Sen1Floods11
s1_after_db, profile = read_s1_rtc_db(after_scene['_item'], bbox=config.DEFAULT_AOI_BBOX)
s1_before_db, _ = read_s1_rtc_db(before_scene['_item'], bbox=config.DEFAULT_AOI_BBOX) if before_scene else (s1_after_db, profile)

train_pairs = read_split_csv(config.SPLITS_DIR / 'flood_train_data.csv')
if train_pairs and (config.S1_HAND_DIR / train_pairs[0][0]).exists():
    ref_chip = load_s1_chip(config.S1_HAND_DIR / train_pairs[0][0])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(ref_chip[1].ravel(), bins=60, alpha=0.6, density=True, label='Sen1Floods11 VH (dB)')
    axes[0].hist(s1_after_db[1].ravel(), bins=60, alpha=0.6, density=True, label='Planetary Computer RTC->dB VH')
    axes[0].set_title('VH Backscatter Histogram Verification'); axes[0].legend()
    axes[1].hist(ref_chip[0].ravel(), bins=60, alpha=0.6, density=True, label='Sen1Floods11 VV (dB)')
    axes[1].hist(s1_after_db[0].ravel(), bins=60, alpha=0.6, density=True, label='Planetary Computer RTC->dB VV')
    axes[1].set_title('VV Backscatter Histogram Verification'); axes[1].legend()
    plt.tight_layout(); plt.show()

In [ ]:
# 3. Sliding-window U-Net inference + JRC permanent water masking + WorldCover & OSM impact
model, mean, std, device = load_trained_unet()
prob_after, raw_mask_after = predict_flood_probability(s1_after_db, model, mean, std, device)
prob_before, raw_mask_before = predict_flood_probability(s1_before_db, model, mean, std, device)

jrc_items = search_auxiliary_item(config.JRC_GSW_COLLECTION, config.DEFAULT_AOI_BBOX, max_items=1)
perm_water = None
if jrc_items:
    occ = reproject_asset_to_grid(jrc_items[0].assets['occurrence'].href, profile)
    perm_water = occ >= 80

flood_mask = clean_flood_mask(raw_mask_after, permanent_water_mask=perm_water)
flood_mask_before = clean_flood_mask(raw_mask_before, permanent_water_mask=perm_water)
flood_gdf = polygonize_mask(flood_mask, profile, prob_raster=prob_after)

wc_items = search_auxiliary_item(config.WORLDCOVER_COLLECTION, config.DEFAULT_AOI_BBOX, max_items=1)
wc_arr = reproject_asset_to_grid(wc_items[0].assets['map'].href, profile) if wc_items else None

impact = compute_impact_metrics(flood_gdf, flood_mask, profile, worldcover_arr=wc_arr, bbox=config.DEFAULT_AOI_BBOX)
print('Bengaluru Flood Prediction (UNVALIDATED):', {k: v for k, v in impact.items() if k != 'assets_gdf'})

In [ ]:
# 4. Sentinel-2 L2A land-change detection (NDVI drop + Bare Soil Index increase)
s2_before_items = search_auxiliary_item(
    config.S2_COLLECTION, config.CHANGE_AOI_BBOX, config.CHANGE_BEFORE_RANGE,
    query={'eo:cloud_cover': {'lt': config.MAX_S2_CLOUD_COVER}}, max_items=5
)
s2_after_items = search_auxiliary_item(
    config.S2_COLLECTION, config.CHANGE_AOI_BBOX, config.CHANGE_AFTER_RANGE,
    query={'eo:cloud_cover': {'lt': config.MAX_S2_CLOUD_COVER}}, max_items=5
)

s2_b_idx, s2_profile = read_s2_l2a_indices(s2_before_items[0], bbox=config.CHANGE_AOI_BBOX)
s2_a_idx, _ = read_s2_l2a_indices(s2_after_items[0], bbox=config.CHANGE_AOI_BBOX)
water_gdf = polygonize_mask(perm_water.astype(np.uint8), profile) if perm_water is not None else None
change_gdf, change_summary = detect_land_change_polygons(s2_b_idx, s2_a_idx, s2_profile, water_gdf=water_gdf)
print('Change summary:', change_summary)

In [ ]:
# 5. Save preview PNGs and export validated JSON + GeoJSON into backend/data/
img_dir = config.BACKEND_IMAGES_DIR
save_sar_preview_png(s1_before_db, img_dir / 's1_before.png', f"S1 Before ({before_scene['datetime'][:10] if before_scene else 'ref'})")
save_sar_preview_png(s1_after_db, img_dir / 's1_after.png', f"S1 After ({after_scene['datetime'][:10]})")
save_sar_preview_png(s1_after_db, img_dir / 'flood_overlay.png', 'Flood Prediction Overlay (Unvalidated)', overlay_mask=flood_mask)
save_rgb_preview_png(s2_b_idx['rgb'], img_dir / 's2_before.png', 'Sentinel-2 Before')
save_rgb_preview_png(s2_a_idx['rgb'], img_dir / 's2_after.png', 'Sentinel-2 After')
save_rgb_preview_png(s2_a_idx['rgb'], img_dir / 'change_overlay.png', 'NDVI Drop + BSI Change Overlay')

model_card = json.loads((config.CHECKPOINTS_DIR / 'model_card.json').read_text())
results_path = export_pipeline_outputs(
    model_card=model_card,
    scenes=flood_scenes,
    flood_gdf=flood_gdf,
    assets_gdf=impact['assets_gdf'],
    change_gdf=change_gdf,
    flood_summary=impact,
    change_summary=change_summary,
    scene_before_id=before_scene['id'] if before_scene else 'N/A',
    scene_after_id=after_scene['id'],
    date_before=before_scene['datetime'][:10] if before_scene else '',
    date_after=after_scene['datetime'][:10],
    change_dates={
        'before': s2_before_items[0].datetime.strftime('%Y-%m-%d'),
        'after': s2_after_items[0].datetime.strftime('%Y-%m-%d'),
    },
)
print('Exported real pipeline results to:', results_path)